# 5G 用户预测 - 人工智能小组作业
## 基于多模型集成的5G用户预测系统

---
**软件工程系 | 第16周提交**

**目标**: 基于用户基本信息和通信数据，使用机器学习算法预测用户是否为5G用户

**评估指标**: AUC

**实现模型**:
- Logistic Regression (基线模型)
- Random Forest
- XGBoost
- LightGBM
- Ensemble (Voting Classifier)

## 1. 环境配置与数据加载

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.', 'libs'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import roc_auc_score, roc_curve
import lightgbm as lgb
import xgboost as xgb

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

RANDOM_STATE = 42
TEST_SIZE = 0.2
print('Environment setup complete!')

In [ ]:
# Load data
df = pd.read_csv('train.csv')
print(f'Dataset shape: {df.shape}')
print(f'Samples: {df.shape[0]}, Features: {df.shape[1] - 2}')
df.head()

## 2. 探索性数据分析 (EDA)

In [ ]:
# Target distribution
target_counts = df['target'].value_counts()
print(f'Non-5G users (0): {target_counts.get(0,0)} ({target_counts.get(0,0)/len(df)*100:.2f}%)')
print(f'5G users (1): {target_counts.get(1,0)} ({target_counts.get(1,0)/len(df)*100:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(['Non-5G', '5G'], [target_counts.get(0,0), target_counts.get(1,0)], color=['#3498db', '#e74c3c'])
axes[0].set_title('Target Distribution'); axes[0].set_ylabel('Count')
axes[1].pie([target_counts.get(0,0), target_counts.get(1,0)], labels=['Non-5G', '5G'],
            autopct='%1.2f%%', colors=['#3498db', '#e74c3c'], explode=(0, 0.05))
axes[1].set_title('Target Proportion')
plt.tight_layout(); plt.savefig('figures/01_target_distribution.png', dpi=150); plt.show()

In [ ]:
# Feature types
cat_cols = [c for c in df.columns if c.startswith('cat_')]
num_cols = [c for c in df.columns if c.startswith('num_')]
print(f'Categorical features: {len(cat_cols)}')
print(f'Numerical features: {len(num_cols)}')

# Missing values
missing = df.isnull().sum().sum()
print(f'Total missing values: {missing}')

In [ ]:
# Correlation analysis
correlations = {}
for c in df.columns:
    if c not in ['id', 'target']:
        try:
            corr = abs(df[c].corr(df['target']))
            if not np.isnan(corr):
                correlations[c] = corr
        except:
            pass

top_features = sorted(correlations.items(), key=lambda x: x[1], reverse=True)[:20]
print('Top 20 features by correlation with target:')
for name, corr in top_features:
    print(f'  {name}: {corr:.4f}')

# Heatmap
top_corr_features = [c for c, _ in top_features[:15]]
corr_matrix = df[top_corr_features + ['target']].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=False, cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Top 15 Feature Correlations', fontsize=14, fontweight='bold')
plt.savefig('figures/02_correlation_heatmap.png', dpi=150); plt.show()

## 3. 数据预处理

In [ ]:
# Prepare data
X = df.drop(columns=['id', 'target'])
y = df['target']

# Handle missing values
if X.isnull().sum().sum() > 0:
    for c in cat_cols:
        X[c] = X[c].fillna(X[c].mode()[0] if len(X[c].mode()) > 0 else -1)
    for c in num_cols:
        X[c] = X[c].fillna(X[c].median())

# Label encode categorical features
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    label_encoders[c] = le

# Winsorize numerical features
for c in num_cols:
    lower = X[c].quantile(0.01)
    upper = X[c].quantile(0.99)
    X[c] = X[c].clip(lower, upper)

# Feature selection: remove low-quality features
# Drop constant numerical features
constant_nums = [c for c in num_cols if X[c].nunique() <= 1]
if constant_nums:
    print(f'Removing {len(constant_nums)} constant numerical features: {constant_nums}')
# Remove low-correlation features (including NaN for constant cols)
low_corr_features = [k for k, v in correlations.items() if v < 0.0005 or np.isnan(v)]
features_to_drop = list(set(constant_nums + [f for f in low_corr_features if f in X.columns]))
features_to_drop = [f for f in features_to_drop if f in X.columns]
if features_to_drop:
    print(f'Removing {len(features_to_drop)} low-quality features')
    X = X.drop(columns=features_to_drop)
    cat_cols = [c for c in cat_cols if c in X.columns]
    num_cols = [c for c in num_cols if c in X.columns]

print(f'Final features: {X.shape[1]}')

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}')

## 4. 模型训练

In [ ]:
results = {}

# Model 1: Logistic Regression (baseline)
print('[1/5] Logistic Regression...')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])
lr.fit(X_train, y_train)
results['Logistic Regression'] = roc_auc_score(y_val, lr.predict_proba(X_val)[:, 1])
print(f'  AUC: {results["Logistic Regression"]:.6f}')

In [ ]:
# Model 2: Random Forest
print('[2/5] Random Forest...')
rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)
results['Random Forest'] = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
print(f'  AUC: {results["Random Forest"]:.6f}')

In [ ]:
# Model 3: XGBoost
print('[3/5] XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
    n_jobs=-1, eval_metric='auc'
)
xgb_model.fit(X_train, y_train, verbose=False)
results['XGBoost'] = roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1])
print(f'  AUC: {results["XGBoost"]:.6f}')

In [ ]:
# Model 4: LightGBM
print('[4/5] LightGBM...')
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=10, learning_rate=0.03,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
)
lgb_model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              eval_metric='auc',
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
results['LightGBM'] = roc_auc_score(y_val, lgb_model.predict_proba(X_val)[:, 1])
print(f'  AUC: {results["LightGBM"]:.6f}')

In [ ]:
# Model 5: Ensemble (Voting)
print('[5/5] Ensemble (Voting)...')
voting = VotingClassifier(
    estimators=[
        ('xgb', xgb.XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='auc')),
        ('lgb', lgb.LGBMClassifier(n_estimators=500, max_depth=10, learning_rate=0.03, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1))
    ],
    voting='soft', weights=[2, 2, 1]
)
voting.fit(X_train, y_train)
results['Ensemble (Voting)'] = roc_auc_score(y_val, voting.predict_proba(X_val)[:, 1])
print(f'  AUC: {results["Ensemble (Voting)"]:.6f}')

## 5. 模型评估与可视化

In [ ]:
# Results summary
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
print('\nAUC Results Summary:')
print(f'{"Model":<25} {"AUC":<12}')
print('-' * 40)
for name, auc in sorted_results:
    print(f'{name:<25} {auc:.6f}')
print(f'\nBest model: {sorted_results[0][0]} (AUC = {sorted_results[0][1]:.6f})')

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
models_dict = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb_model, 'LightGBM': lgb_model, 'Ensemble': voting}
for i, (name, model) in enumerate(models_dict.items()):
    y_prob = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{name} (AUC={results[name]:.5f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3); ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.savefig('figures/03_roc_curves.png', dpi=150); plt.show()

In [ ]:
# AUC Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
names_list = [n for n, _ in sorted_results]
aucs_list = [a for _, a in sorted_results]
ax.barh(names_list, aucs_list, color=['#e74c3c' if i == 0 else '#3498db' for i in range(len(names_list))])
for i, (bar, val) in enumerate(zip(ax.patches, aucs_list)):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.5f}', va='center', fontweight='bold')
ax.set_xlabel('AUC Score', fontsize=12)
ax.set_title('Model AUC Comparison', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.savefig('figures/04_auc_comparison.png', dpi=150); plt.show()

In [ ]:
# Feature Importance
feature_importances = pd.Series(lgb_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 8))
top_fi = feature_importances.head(30)
ax.barh(range(30), top_fi.values, color=plt.cm.RdYlGn(np.linspace(0.2, 0.8, 30))[::-1])
ax.set_yticks(range(30)); ax.set_yticklabels(top_fi.index); ax.invert_yaxis()
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top 30 Feature Importances (LightGBM)', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.savefig('figures/05_feature_importance.png', dpi=150); plt.show()

## 6. 交叉验证

In [ ]:
# K-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_models = {
    'LightGBM': lgb.LGBMClassifier(n_estimators=300, max_depth=10, learning_rate=0.03, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, eval_metric='auc'),
}

for name, model in cv_models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
    print(f'{name}: CV AUC = {cv_scores.mean():.5f} +/- {cv_scores.std():.5f}')

## 7. 功能拓展 - 超参数优化

In [ ]:
# Grid Search for XGBoost
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.03, 0.05],
}
base_xgb = xgb.XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, eval_metric='auc')
grid_search = GridSearchCV(base_xgb, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print(f'Best params: {grid_search.best_params_}')
print(f'Best CV AUC: {grid_search.best_score_:.5f}')

best_xgb = grid_search.best_estimator_
tuned_auc = roc_auc_score(y_val, best_xgb.predict_proba(X_val)[:, 1])
print(f'Tuned XGBoost Validation AUC: {tuned_auc:.5f}')

## 8. 总结

本实验通过多种机器学习模型对5G用户进行分类预测，主要发现：

1. **数据特点**: 数据集包含60个字段，涵盖离散型和数值型特征，目标变量分布存在一定的不平衡
2. **模型选择**: LightGBM和XGBoost在AUC指标上表现最优，得益于其对类别不平衡的鲁棒性和高效的梯度提升算法
3. **特征重要性**: 部分cat_和num_特征对预测目标贡献较大，体现了用户的通信行为与5G使用意愿之间的关联
4. **集成策略**: Voting集成的软投票策略进一步提升了模型鲁棒性
5. **改进方向**: 可尝试更复杂的特征工程、深度神经网络、以及更大范围的超参数搜索